# Linear Regression AIPM

## First part - Setup and visuals (instructor)

In [ ]:
import numpy as np
import json
import pandas as pd
import seaborn as sns
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

In [ ]:
# Import company colors
with open('../plot_colors.json', 'r') as pc:
    color_dict = json.load(pc)

c_light, c_dark, c_blue = color_dict['color_light'], color_dict['color_dark'], color_dict['color_blue']

# Define color map for plotting
color_map = ListedColormap([c_blue, c_light, c_dark])

### Linear regression equation plot

In [ ]:
def linear_eq(annotate=False):

    plt.rcParams.update({'font.size': 15})

    # Generate x values
    x = np.linspace(0, 10, 100)

    # Calculate y values
    y = 2 + 1/3 * x
    circle1 = plt.Circle((3, 3), 0.15, color='orange', fill=True)
    circle2 = plt.Circle((6, 4), 0.15, color='orange', fill=True)


    plt.gca().add_patch(circle1)
    plt.gca().add_patch(circle2)

    # Create the plot
    plt.plot(x, y)

    # Add labels and title
    plt.xlim(0,7)
    plt.ylim(0.6)
    plt.xlabel('x')
    plt.ylabel('y')
    plt.grid()

    if annotate:
        # Annotate the intercept
        plt.annotate('y-intercept',
                     xy=(0, 2),
                     xytext=(-90, 30),
                     textcoords='offset points',
                     arrowprops=dict(arrowstyle='->'))

        # Annotate general equation
        plt.annotate(r'$ y = b_0 + b_1*x $',
                     xy=(7, 4.2),
                     xytext=(5, 5),
                     textcoords='offset points')

        # Annotate slope
        plt.annotate(r'$ b_1 = \frac{\Delta y}{\Delta x}$',
                     xy=(7, 2.5),
                     xytext=(5, 5),
                     textcoords='offset points')

        # Annotate equation
        plt.annotate(r'Equation : $ y = 2 + \frac{1}{3}*x$',
                     xy=(7, 0.8),
                     xytext=(8, 8),
                     textcoords='offset points')
        
        # Annotate delta x and y arrows
        arrowstyle1 = FancyArrowPatch((6, 3), (6, 4),
                                 connectionstyle="arc3,rad=0.2", arrowstyle='<->, head_width=0.3',
                                 lw=1.5, mutation_scale=10, color='black')
        plt.gca().add_patch(arrowstyle1)
        
        arrowstyle2 = FancyArrowPatch((3, 3), (6, 3),
                                 connectionstyle="arc3,rad=0.2", arrowstyle='<->, head_width=0.3',
                                 lw=1.5, mutation_scale=10, color='black')
        
        plt.gca().add_patch(arrowstyle2)
        
        plt.annotate(r'$ \Delta y $',
             xy=(6.1, 3.3),
             xytext=(6.1, 3.3),
             textcoords='offset points')
        plt.annotate(r'$ \Delta x $',
             xy=(4.1, 2.3),
             xytext=(4.1, 2.3),
             textcoords='offset points')

    # Display the plot
    #plt.show()

### Linear regression example plot

In [ ]:
def generate_data(num_points):
    np.random.seed(4)
    x = np.random.uniform(low=0, high=12, size=num_points)
    noise = np.random.normal(loc=0, scale=0.8, size=num_points)
    y = 0.3 + 0.5 * x + noise
    x = x[y > 0]
    y = y[y > 0]
    X = np.column_stack((np.ones(len(x)), x))
    feature_columns = pd.Index(['intercept', 'Week of bootcamp'])
    X_df = pd.DataFrame(X, columns=feature_columns)
    X_df['Coffee consumption in [l]'] = y
    return X_df, X, y.reshape(-1, 1)

num_points=50
data, X, y = generate_data(num_points)


In [ ]:
data_lines = data.copy()

x_mean = data_lines['Week of bootcamp'].mean()
y_mean = data_lines['Coffee consumption in [l]'].mean()

reg_line_columns = pd.Index([
    'b1',
    'b0',
    'p_at_0',
    'p_at_12',
    'sum_of_squared_residuals',
    'sum_of_residuals',
])
reg_lines = pd.DataFrame(columns=reg_line_columns)

for b1 in [0.1, 0.3, 0.4, 0.5, 0.6, 0.8, 1]:
    # for each scope calculate a so every line goes through the mean of the data points
    b0 = y_mean - b1*x_mean
    
    # getting the plotting points at x = 1 and x = 11 (to define a line)
    p1 = b0 + b1*0
    p2 = b0 + b1*12
    
    data_lines.loc[:,f'prediction_b_{b1}'] = b0 + b1 * data_lines['Week of bootcamp']
    data_lines[f"residuals_{b1}_sq"]= (data_lines['Coffee consumption in [l]'] - data_lines[f'prediction_b_{b1}'])**2
    data_lines[f"residuals_{b1}"]= (data_lines['Coffee consumption in [l]'] - data_lines[f'prediction_b_{b1}'])
    ssr = sum(data_lines[f"residuals_{b1}_sq"])
    sr = sum(data_lines[f"residuals_{b1}"])
    
    reg_lines.loc[len(reg_lines.index)] = [b1, b0, p1, p2,  ssr, sr]


### Plot data

In [ ]:
def plot_data(x, y, reg_data=None, figsize=(10,6), regression_line=False, residuals=False, line_selection=None, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    
    sns.set_context("talk", font_scale=0.8, rc={"lines.linewidth": 2.})
    
    # Plot scatter and regression line if set to True
    sns.regplot(x=x, y=y, scatter_kws={'s':30, "alpha": 1, "color": c_blue},
                ci=None, color=c_light, fit_reg=regression_line, ax=ax)
    
    if residuals:
        b1, b0 = np.polyfit(x, y, 1)
        regression_line = b0 + b1*x
        for i in range(len(x)):
            ax.plot([x[i], x[i]], [y[i], regression_line[i]], color=c_dark, linewidth=1)
            
    if isinstance(reg_data, pd.DataFrame):
        colors=["#191970", "#3765b5", "#617baa", "#bac2ca", "#617baa", "#191970", "#570f4c"]
        if line_selection is not None:
            for j, line in reg_data.iterrows():
                if j in line_selection:
                    ax.plot([0, 12], [line["p_at_0"], line["p_at_12"]], color=colors[j], linewidth=2)
        else:
            for j, line in reg_data.iterrows():
                ax.plot([0, 12], [line["p_at_0"], line["p_at_12"]], color=colors[j], linewidth=2)

    
    ax.set_xlabel('Week of bootcamp')
    ax.set_ylabel('Coffee consumption [l]')
    sns.despine()

### Fitted lines with residuals plot

In [ ]:
def plot_possible_lines_with_residuals(x, y, reg_data=None, figsize=(10,6), line_index=3, residuals=False, ssr=False, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    
    sns.set_context("talk", font_scale=0.8, rc={"lines.linewidth": 2.})
    
    # Plot scatter and regression line if set to True
    sns.regplot(x=x, y=y, scatter_kws={'s':30, "alpha": 1, "color": c_blue},
                ci=None, color=c_light, fit_reg=False, ax=ax)
    
    #colors = ['#570f4c', 'darkblue', '#bac2ca', 'grey', '#191970', '#907390', '#570f4c']
    colors=["#191970", "#3765b5", "#617baa", "#bac2ca", "#617baa", "#191970", "#570f4c"]
    
    ax.plot([0, 12],[reg_lines["p_at_0"][line_index], reg_lines["p_at_12"][line_index]], color=colors[line_index], linewidth=2)
    
    if residuals:
        if not isinstance(reg_data, pd.DataFrame):
            raise ValueError('reg_data must be a pandas DataFrame when residuals is True.')
        regression_line = reg_data.loc[line_index, 'b0'] + reg_data.loc[line_index, 'b1']*x
        for i in range(len(x)):
            ax.plot([x[i], x[i]], [y[i], regression_line[i]], color=c_dark, linewidth=1)
            
    if ssr:
        if not isinstance(reg_data, pd.DataFrame):
            raise ValueError('reg_data must be a pandas DataFrame when ssr is True.')
        ax.annotate(text=f"SSR = {reg_data.loc[line_index, 'sum_of_squared_residuals'].round(2)}",
        xy=(0, 7),
        xytext=(1, 2),
        textcoords='offset points')
            
    ax.set_xlabel('Week of bootcamp')
    ax.set_ylabel('Coffee consumption [l]')
    sns.despine()


### Additional linear-regression visualizations


In [ ]:
def calc_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    residuals = y_true - y_pred
    mse = np.mean(residuals**2)
    rmse = np.sqrt(mse)
    sst = np.sum((y_true - y_true.mean())**2)
    r2 = 1 - np.sum(residuals**2) / sst
    return rmse, r2

def plot_baseline_vs_regression(x, y, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 5))

    x = np.asarray(x).reshape(-1)
    y = np.asarray(y).reshape(-1)

    sns.scatterplot(x=x, y=y, s=35, color=c_blue, alpha=0.95, ax=ax)

    y_mean = y.mean()
    ax.axhline(y_mean, color=c_dark, linestyle='--', linewidth=2, label='Mean baseline')

    b1, b0 = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 200)
    ax.plot(x_line, b0 + b1 * x_line, color=c_light, linewidth=3, label='Fitted regression')

    ax.set_xlabel('Week of bootcamp')
    ax.set_ylabel('Coffee consumption [l]')
    ax.set_title('Baseline (mean) vs fitted regression')
    ax.legend(frameon=False)
    sns.despine()


def plot_metric_comparison_for_two_lines(x, y, reg_data, line_a=1, line_b=3, ax=None):
    if ax is None:
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    x = np.asarray(x).reshape(-1)
    y = np.asarray(y).reshape(-1)

    labels = ['Line A', 'Line B']
    idxs = [line_a, line_b]

    rmse_vals, r2_vals = [], []
    for idx in idxs:
        b0 = reg_data.loc[idx, 'b0']
        b1 = reg_data.loc[idx, 'b1']
        pred = b0 + b1 * x
        rmse, r2 = calc_regression_metrics(y, pred)
        rmse_vals.append(rmse)
        r2_vals.append(r2)

    colors = [c_dark, c_light]

    ax[0].bar(labels, rmse_vals, color=colors)
    ax[0].set_title('RMSE (lower is better)')
    ax[0].set_ylabel('RMSE')

    ax[1].bar(labels, r2_vals, color=colors)
    ax[1].set_title('$R^2$ (higher is better)')
    ax[1].set_ylabel('$R^2$')
    ax[1].set_ylim(0, 1)

    sns.despine()



## Second part - PM-focused lecture

<div class="slide-title"> 
        
# Linear Regression
           
</div> 

Notes:
- Frame this as a decision-making tool for product work, not a math lecture.
- Promise practical outcomes: when to use linear regression, when not to use it, and how to interpret outputs safely.
- Keep formulas lightweight and spend most time on business interpretation.


### Historical Origin: Regression to the Mean

<div class="group">
  <div class="text">

* In the 19th century, Francis Galton studied parent and child heights.
* He observed that extreme values tend to be followed by values closer to the average.
* He called this pattern <i>regression toward mediocrity</i> (now: regression to the mean).

<div class="alert alert-block alert-info">
<b>Why this matters:</b><br>
The term <b>regression</b> originally came from this statistical phenomenon, before modern ML usage.
</div>

  </div>
  <div class="images">
    <img src="../../images/linear_regression/francis_galton_1850s.jpg" width="320"><br>
    <small>Francis Galton (via Wikipedia/Wikimedia Commons)</small>
  </div>
</div>



Notes:
- Keep this short (historical context, not deep theory).
- Core intuition: extreme observations are often followed by less extreme ones.
- Clarify that this does not imply causality; it is a statistical tendency.
- Transition: today we use regression to model and predict continuous outcomes.


## Why PMs should care

How linear regression supports product planning, prioritization, and stakeholder communication.


Notes:
- Start with familiar PM questions: "What will signups be next month?" "How much does response time affect conversion?"
- Position linear regression as a strong baseline: fast, interpretable, and useful for first-pass decisions.
- Emphasize that PM value comes from asking the right question and interpreting model output in context.


### Goal of regression in product work

<div class="group">
  <div class="text">

* You are planning next quarter for a B2B SaaS product.
* You want to estimate monthly trial signups from drivers like ad spend, seasonality, and sales outreach.
* You need a model that is simple enough to explain to stakeholders and useful enough for planning.

  </div>
  <div class="images">
    <img src="../../images/linear_regression/img_p3_1.png">
  </div>

</div>


Notes:
- Map the story to ML terms: target = trial signups (or revenue), features = potential drivers.
- Clarify that the model supports decisions, but does not replace PM judgment.
- Encourage discussion on data quality: are these features measurable and trustworthy?


### How could a PM estimate outcomes?

<div class="group">
  <div class="text">

* Option A: pick one similar historical month
* Quick, but brittle and hard to generalize
  </div>
  <div class="text">
* Option B: learn a general relationship from many months
* More reliable for planning and scenario analysis
  </div>
</div>


Notes: A regression model gives a reusable rule that links business inputs (x) to an expected outcome (y).


### Example: a PM-facing regression rule

```
predicted_monthly_signups =
    1200                    # baseline demand
  + 4.5 * ad_spend_k        # +4.5 signups per additional 1k EUR spend
  - 35  * avg_response_hr   # slower responses hurt conversion
  + 90  * campaign_flag     # effect of campaign period (0/1)
```

<div class="alert alert-block alert-info">
<b>PM interpretation:</b><br>
Each coefficient is an estimated directional effect, assuming other variables stay constant.
</div>


Notes: The formula is not "truth"; it is a compact summary of historical patterns and should be validated against business reality.


### Building a model for PM decisions

<div class="group">
  <div class="text">
  <h4> Explanation (insight) </h4>
  Use linear regression to understand drivers

  -> Which factors are most associated with movement in the KPI?

  -> Which levers look worth testing further?

  <h4> Prediction (planning) </h4>
  Use linear regression to forecast likely outcomes

  -> What should we expect next month?

  -> What happens if we change one lever?


  </div>
  <div class="images">
    <img src="../../images/linear_regression/img_p6_1.png">
  </div>

</div>



Notes: Keep both modes clear.
- Explanation supports prioritization and hypothesis generation.
- Prediction supports planning and expectation setting.
- Neither mode alone proves causality.



## Intuition: the line behind the model

Notes:
- Explain the line only as far as needed for interpretation.
- Intercept = baseline level; slope = expected KPI change for +1 unit in a feature.
- Do not spend time deriving formulas from two points.


### The core idea

A linear model combines a baseline plus weighted feature effects.

Notes:
- Ask: "If this feature increases by 1 unit, how much does prediction move?"
- This is the PM-friendly meaning of a coefficient.
- Mention units explicitly to avoid misinterpretation.


In [ ]:
linear_eq()

### Linear Equation


In [ ]:
linear_eq(annotate=True)

<div class="alert alert-block alert-info">
<b>Key terms:</b>
<ul>
  <li>Intercept (<i>b</i><sub>0</sub>): predicted value of <i>y</i> when <i>x</i> = 0.</li>
  <li>Slope (<i>b</i><sub>1</sub>): expected change in <i>y</i> when <i>x</i> increases by 1 unit.</li>
</ul>
</div>






Notes: Keep this visual and practical. No manual derivation needed for this cohort.


## From line to real-world prediction

Notes:
- Transition: line equation was geometry, now we apply it to noisy real data.
- In practice, points are not exactly on one line, so we need a fitting rule.


### Linear regression in PM terms

* Question 1: Which product or business signals move with our KPI?

* Question 2: Can we use those signals to make useful forecasts?

<div class="alert alert-block alert-info">
<b>Important:</b><br>
Correlation and regression show association, not guaranteed causation. Use experiments or quasi-experiments for causal claims.
</div>





Notes:
- Reinforce decision framing: is this model useful enough to support planning?
- Prompt discussion on confounders and omitted variables.
- Set expectation: model quality is judged by out-of-sample performance, not just in-sample fit.
- Quasi-experiments - estimates causal impacts of interventions by comparing groups without using random assignment. They resemble true experiments but often use pre-existing groups, making them ideal for real-world settings where randomization is impractical or unethical


### What about more than 2 points?

<center>
<img src="../../images/linear_regression/img_p14_1.png" width=600>  
</center>

Notes:  
Let students suggest some regressions (draw into zoom window).  
Would a non-linear model be sensible?  
Are the suggestions of different quality?  
What is the line good for?   
$\rightarrow$ we need a quantitative measure to decide what is "good".


### Example dataset

<div class="group">
  <div class="text">

      
* week of bootcamp, $x$
* coffee consumption, $y$
* clear positive relationship


  </div>

  <div class="images">

$y = b_{0}+b_{1}\cdot x+e$

$\rightarrow$ Find the line that predicts best on average
  </div>
</div>






Notes:
- Be explicit that coffee is only a didactic example; transfer the idea to product KPIs.
- Ask learners to suggest PM analogs (conversion rate, churn, retention, demand).
- Keep notation light and focus on intuition.



In [ ]:
plot_data(data['Week of bootcamp'], data['Coffee consumption in [l]'], figsize=(8,5))

### Two candidate models: which one should a PM trust?

* Model A and Model B both look plausible visually.
* We need a quantitative criterion, not just intuition.

<div class="alert alert-block alert-info">
<b>Note:</b> <br>
$\hat{y}$ (y-hat) is the model prediction, not an observed value.
</div>






In [ ]:
plot_data(data['Week of bootcamp'], data['Coffee consumption in [l]'], figsize=(8,5), reg_data=reg_lines, line_selection=[1, 3])

Notes:
- Remind students that the line is the model and the points are reality.
- The model deviates from the data, not the other way around.
- Residuals measure that deviation.


### How do we know which line is better?

Notes:
- Ask students to vote which line looks better before showing formulas.
- Use that moment to motivate the need for an objective numeric criterion.


### Residuals (prediction errors)

For each observation $i$:

$$e_i = y_i - \hat{y}_i$$

PM interpretation:
- Residual tells how wrong the model is for one point.
- Positive residual: model predicted too low.
- Negative residual: model predicted too high.
- Residual size indicates miss severity.




Notes:
- Define residual as: observed minus predicted.
- Keep sign interpretation explicit: positive = underestimation, negative = overestimation.
- Link formula to the vertical distance shown in the plot.


### Residual example

Suppose one observation has:
- Observed value: $y = 4.8$
- Predicted value: $\hat{y} = 3.3$

Then:
$$e = 4.8 - 3.3 = 1.5$$

Meaning: the model underpredicted by 1.5 units.


Notes:
- Walk through this slowly and write each number only once.
- Emphasize the order: residual = real minus predicted.
- Ask students what the sign means before revealing interpretation.
- Repeat: positive residual means the point lies above the line.


In [ ]:
plot_data(data['Week of bootcamp'], data['Coffee consumption in [l]'], figsize=(8,5), regression_line=True, residuals=True)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15,5))
plot_possible_lines_with_residuals(data['Week of bootcamp'], data['Coffee consumption in [l]'], reg_lines, line_index=1, residuals=True, ax=ax[0])
plot_possible_lines_with_residuals(data['Week of bootcamp'], data['Coffee consumption in [l]'], reg_lines, line_index=3, residuals=True, ax=ax[1])

### How we score a line: least squares

To compare candidate lines, we use one error score:

$$ J = \sum_{i=1}^{n}(y_{i} - \hat{y}_{i})^{2} $$

How to read this:
- Compute error at each point.
- Square errors (large misses hurt more).
- Sum them into one total score.
- Lower $J$ means better fit on this data.






In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15,5))
plot_possible_lines_with_residuals(data['Week of bootcamp'], data['Coffee consumption in [l]'], 
                                   reg_lines, line_index=1, residuals=True, ssr=True, ax=ax[0])
plot_possible_lines_with_residuals(data['Week of bootcamp'], data['Coffee consumption in [l]'], 
                                   reg_lines, line_index=3, residuals=True, ssr=True, ax=ax[1])

Notes:
- Explain why we square errors: no cancellation and stronger penalty on large misses.
- Keep optimization details minimal: libraries find the best coefficients automatically.
- PM takeaway: compare models using robust validation metrics, not by eye.


### Trying several fitted lines

By comparing the sum of squared residuals, we can decide which line fits better:

$$ J(b_{0}, b_{1}) = \sum_{i=1}^{n} e_{i}^{2} = \sum_{i=1}^{n}(y_{i} - \hat{y}_{i})^{2} = \sum_{i=1}^{n}(y_{i} - (b_{0} + b_{1}x_{i}))^{2} $$

Beginner view:
- Each candidate line gets one score: $J$
- Lower score = line stays closer to points overall.




In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
plot_data(
    data['Week of bootcamp'],
    data['Coffee consumption in [l]'],
    reg_data=reg_lines,
    residuals=False,
    ax=ax
)


Notes:
- Treat `J` as a "badness score" for the line.
- Changing slope/intercept changes this score.
- Goal: find parameters with the smallest possible score.
- For one feature, we can derive a closed-form solution.
- For many features, numerical optimization (for example, gradient descent) is often preferred.


### There are infinitely many possible lines


Notes: What does this mean?
We can evaluate how good a line is, but checking every possible line is not feasible.

The following slides can be shown briefly. The key message is that the objective function is a quantitative value we can optimize by choosing suitable `b0` and `b1` (later: Gradient Descent).


### So how is the model fitted?

* We search for coefficient values that minimize prediction error.
* In practice, libraries perform this optimization automatically.
* Your PM role is to validate assumptions, data quality, and decision usefulness.

$$ J = \sum (y_{i} - \hat{y}_{i})^{2} $$

Plain language: pick the model with the lowest validated error, not the prettiest equation.






Notes:
- "Minimize" means: search for the smallest possible loss.
- `b0` shifts the line up/down.
- `b1` tilts the line.
- Together they control predictions and therefore residuals.


### Quick residual sanity checks

After fitting, check whether errors look random (not patterned).

Practical checks:
- Residuals should be centered around zero.
- No obvious trend of residuals versus fitted values.
- Large structured errors suggest missing variables or wrong model form.


<div class="alert alert-block alert-info">
<b>PM takeaway:</b> If residuals have structure, your model is missing part of the story.
</div>


Notes:
- Explain why these residual properties are useful checks for OLS.
- Intuition: over- and under-predictions balance out around zero.
- Residuals should not keep a linear pattern with `x` after fitting.


## Model performance

Notes:
- Reconnect to prior knowledge: average and spread.
- Mean = baseline prediction before using any feature.
- Variance = how much values naturally differ around the mean.


### Baseline first, model second

Before modeling, define a baseline prediction (often the mean).

Then ask:
- Does regression materially beat this baseline?
- Is the improvement large enough to influence decisions?



Notes:
- Read both formulas slowly and define each symbol.
- Explain why variance uses squared distance from the mean.
- Clarify that `n - 1` is a degrees-of-freedom correction (sample variance).
- Pronunciation guide:
  * $\sigma^2$ = "sigma squared"
  * $\bar{y}$ = "y-bar"


### Baseline Mean vs Fitted Regression

A constant mean line is a simple baseline model.
A fitted regression line should reduce error compared with this baseline.


In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
plot_baseline_vs_regression(
    data['Week of bootcamp'],
    data['Coffee consumption in [l]'],
    ax=ax
)


Notes:
- The dashed line is the baseline: always predict the mean.
- The fitted line uses x-information and should follow the data better.
- This sets up why $R^2$ compares explained vs unexplained variance.


<div class="group">
  <div class="images">
    <img src="../../images/linear_regression/mean_variance.png" width="1000">
  </div>

</div>

Notes:
- Baseline model: predict `y_bar` for everyone.
- If data points are very spread out, baseline errors are large.
- A good regression model should reduce error compared with this baseline.


### Sum of Squares Decomposition (Variance Analysis)

<div class="images">
  <img src="../../images/linear_regression/variance_analysis.png" width="1000"><br>
</div>


### Explained vs unexplained variation

$$SST = SSE + SSR$$

Interpretation:
- $SST$: total variation in the target KPI
- $SSE$: variation explained by the model
- $SSR$: variation left unexplained

<div class="alert alert-block alert-info">
<b>PM reading:</b> better models reduce unexplained variation, but some uncertainty always remains.
</div>

Notes:
- Start from baseline variability (SST).
- Regression splits it into:
  * explained part (SSE)
  * unexplained part (SSR)
- If SSR gets smaller, predictions improved.
- Then $R^2$ gets larger because more variance is explained.


### RMSE (average prediction miss)

$$ RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}\left(y_{i}-{\hat{y}}_{i}\right)^{2}} $$

PM interpretation:
- RMSE is the typical size of prediction error.
- It uses the same unit as the KPI (easy to explain).
- Lower RMSE is better, but always compare to baseline and business tolerance.








Notes:
- Emphasize workflow: residuals $\rightarrow$ square $\rightarrow$ average $\rightarrow$ square root.
- RMSE is easy to interpret because it uses the original target unit.
- Lower RMSE indicates better average predictive accuracy.



### Which metric answers which PM question?

- <b>RMSE</b>: "How wrong are we on average?"
  Use when planning risk and setting forecast confidence.

- <b>$R^2$</b>: "How much KPI variation is captured?"
  Use for model explanatory power at a high level.

- <b>Adjusted $R^2$</b>: "Did adding more features actually help?"
  Use to avoid complexity without real gain.



Notes:
- Give one plain-language question per metric.
- Suggest students report RMSE and adjusted $R^2$ together.
- RMSE gives practical error size; adjusted $R^2$ gives model-complexity awareness.
- Avoid claiming one metric is always enough.


### Metric Comparison for Two Candidate Lines

Same dataset, two fitted lines.
Use <b>RMSE</b> and $R^2$ to compare their quality directly.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
plot_metric_comparison_for_two_lines(
    data['Week of bootcamp'],
    data['Coffee consumption in [l]'],
    reg_lines,
    line_a=1,
    line_b=3,
    ax=ax
)

Notes:
- One model can look similar visually; metrics make the comparison explicit.
- Lower RMSE and higher $R^2$ should point to the better-fitting line.
- Reinforce: use multiple metrics, not only one number.


## PM playbook

Notes:
- Transition into terminology consolidation.
- Encourage students to focus on meaning, not memorizing every symbol at once.


### PM checklist: when linear regression is a good choice

Use linear regression when:
- You predict a continuous KPI (e.g., revenue, demand, time, spend).
- You need interpretable driver effects for stakeholder discussions.
- Relationships are approximately linear in the relevant range.
- You need a fast baseline before trying more complex models.

Be careful when:
- Outcome is categorical (use classification methods instead).
- Strong non-linearity dominates and linear approximation fails.
- Data has major missing drivers, leakage, or unstable definitions.


Notes: Clarify terminology. Different fields (statistics, engineering, economics) often use different names for the same concepts.


### What about more than one feature?

Notes:
- Transition from one-feature regression to real-world multi-feature models.
- Reassure students: same core idea, just matrix notation for efficiency.


## Multiple Linear Regression

Notes:
Transition point:
- Simple linear regression used one feature (`x`).
- Multiple linear regression uses many features (`x1`, `x2`, ..., `xm`).
- We keep one target variable for now.


### Multiple linear regression (concept)

Same idea, more drivers:

$$\hat{y} = b_0 + b_1x_1 + b_2x_2 + ... + b_mx_m$$

PM interpretation:
- Each coefficient estimates directional effect, holding other included variables constant.
- Coefficients help prioritize levers, but they do not prove causality.
- More features can improve fit, but also increase risk of overfitting and noise.




Notes (what to explain on this slide):
- Keep notation minimal: outcome equals baseline plus weighted feature effects.
- Focus on interpretation: each weight describes expected direction and magnitude, all else equal.
- Remind learners that coefficient size depends on feature units and scaling.
- For PM decisions, combine coefficients with domain context, not in isolation.



Notes (quick PM example):
- Forecast weekly churn with features: support backlog, bug count, and average response time.
- Each row is one week; each column is one potential driver.
- Output is one predicted churn value per week.


### Fitting in practice

Whether using one feature or many, modern libraries estimate coefficients for you.

What matters for PMs:
- Define the decision question clearly.
- Check stability over time and by segment.
- Communicate uncertainty, not only point predictions.




Notes:
- Avoid deep matrix algebra.
- Emphasize that implementation details are delegated to libraries.
- Keep learner attention on data assumptions, metric trade-offs, and business decisions.


### Multiple-feature models: evaluating usefulness

Use two complementary views:

- <b>RMSE</b>: average miss size in KPI units (lower is better)
- <b>Adjusted $R^2$</b>: did added features create real signal or just complexity?

<div class="alert alert-block alert-info">
<b>Practical rule:</b>
Prefer the simpler model unless the more complex one gives a clear, stable gain on validation data.
</div>







Notes:
- Ask: "Would this improvement change a real product decision?"
- If not, favor simpler models for communication and maintenance.
- Always evaluate on holdout periods, not only historical fit.


### PM summary: what to remember

- Linear regression predicts continuous outcomes using weighted inputs.
- Coefficients are interpretable estimates, not automatic causal truths.
- Residuals reveal where and how the model fails.
- RMSE communicates typical miss size; adjusted $R^2$ helps manage complexity.
- Baselines and validation are mandatory before trusting forecasts.


<div class="alert alert-block alert-info">
<b>Decision lens:</b>
Use the model when it improves planning quality and can be explained clearly to stakeholders.
</div>







Notes:
- Run this as a recap conversation, not a memorization check.
- Ask each group to share one safe use case and one risky misuse of linear regression.
- Close on model governance: assumptions, drift checks, and stakeholder alignment.


## References

Suggested resources for deeper understanding and practical application:

[Practical Statistics for Data Science](https://www.oreilly.com/library/view/practical-statistics-for/9781491952955/) - Peter Bruce & Andrew Bruce

[Written explanation of linear regression](https://learningstatisticswithr.com/book/regression.html)

[Difference between $R^2$ and adjusted $R^2$](https://www.investopedia.com/ask/answers/012615/whats-difference-between-rsquared-and-adjusted-rsquared.asp)

[StatQuest: Linear Regression](https://www.youtube.com/watch?v=7ArmBVF2dCs)



Notes:
- End with action items: define one PM question, one baseline, and one evaluation metric plan.
- Encourage learners to pair with DS/analytics to test one real product hypothesis this week.
